# PyTorch Coding Drills

Timed practice for implementing ML components under pressure. Format: single problem, ~35 minutes of coding followed by 5-10 minutes of review.

## How to use this notebook

1. Read the problem statement in each drill's markdown cell.
2. **Set a 35-minute timer.**
3. Open a blank scratch file and implement the solution from scratch — do not look ahead.
4. Only check the reference solution cell after your timer expires.
5. Compare your approach, note gaps, and repeat the drill until you can complete it cleanly within time.

## Environment notes

- All drills are primarily CPU-based. VRAM usage if you test on GPU: ~2-4 GB.
- Required imports per drill are listed in the problem statement.
- Solutions are self-contained and runnable as-is.

## Drill 1: Multi-Head Attention from Scratch

**Time target: 35 minutes**

### Problem

Implement a `MultiHeadAttention` class as an `nn.Module` from scratch in PyTorch.

**Interface:**
```python
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int) -> None: ...
    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor: ...
```

**Requirements:**
- Input shape: `(batch_size, seq_len, embed_dim)`, optional `mask` tensor
- Parameters: `embed_dim`, `num_heads`
- Must include: Q/K/V linear projections, scaled dot-product attention, optional mask support, output projection
- **Constraint:** No `nn.MultiheadAttention`. Only use `nn.Linear`, `nn.Module`, basic tensor ops, and `F.softmax`.

**Test your implementation:**
```python
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x = torch.randn(2, 10, 64)
out = mha(x)  # should be (2, 10, 64)
```
Also test with a causal mask and verify the backward pass runs without errors.

**Imports needed:** `torch`, `torch.nn`, `torch.nn.functional`, `math`

In [ ]:
# YOUR IMPLEMENTATION — attempt this before looking at the solution below
# Set a timer. When it rings, stop and check the solution.

raise NotImplementedError


In [ ]:
# === SOLUTION === (only look after attempting above) ===

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MultiHeadAttention(nn.Module):
    """Multi-head self-attention implemented from scratch.

    Args:
        embed_dim: Total embedding dimension.
        num_heads: Number of attention heads. embed_dim must be divisible by num_heads.
    """

    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        """Compute multi-head attention.

        Args:
            x: Input tensor of shape (B, S, D).
            mask: Optional mask tensor of shape (1, 1, S, S) or (B, 1, S, S).
                  Positions where mask == 0 are masked out (set to -inf).

        Returns:
            Output tensor of shape (B, S, D).
        """
        B, S, D = x.shape

        # Project and reshape to (B, num_heads, S, head_dim)
        Q = self.W_q(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention: (B, H, S, S)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_output = attn_weights @ V  # (B, H, S, head_dim)

        # Merge heads back: (B, S, D)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(attn_output)


# --- Tests ---
mha = MultiHeadAttention(embed_dim=64, num_heads=8)
x = torch.randn(2, 10, 64)

out = mha(x)
assert out.shape == (2, 10, 64), f"Expected (2, 10, 64), got {out.shape}"

# Causal (lower-triangular) mask
causal_mask = torch.tril(torch.ones(10, 10)).unsqueeze(0).unsqueeze(0)  # (1, 1, 10, 10)
out_masked = mha(x, mask=causal_mask)
assert out_masked.shape == (2, 10, 64)

# Backward pass
loss = out_masked.sum()
loss.backward()

print(f"Output shape: {out.shape}")
print(f"Param count: {sum(p.numel() for p in mha.parameters()):,}")
print("Forward + backward pass: OK")

## Drill 2: DDPM Forward Process + Training Step

**Time target: 35 minutes**

### Problem

Implement the DDPM noising process and a single training step.

**Part 1 — Schedule:**
```python
def linear_beta_schedule(timesteps: int) -> torch.Tensor:
    """Returns betas linearly spaced from 1e-4 to 0.02."""
```
Precompute and store: `alphas`, `alpha_cumprod`, `sqrt_alpha_cumprod`, `sqrt_one_minus_alpha_cumprod`.

**Part 2 — Forward diffusion (reparameterization trick):**
```python
def forward_diffusion(x_0: Tensor, t: Tensor, noise: Tensor) -> Tensor:
    """Returns x_t = sqrt(alpha_cumprod_t) * x_0 + sqrt(1 - alpha_cumprod_t) * noise"""
```

**Part 3 — Training step:**
```python
def training_step(model: nn.Module, x_0: Tensor) -> Tensor:
    """Sample random t, noise x_0, predict noise, return MSE loss."""
```

**Test with:**
- `x_0` of shape `(4, 3, 32, 32)`
- A dummy model that accepts `(x: Tensor, t: Tensor)` and returns same shape as `x`
- Assert `x_t` shape matches `x_0`, assert loss is a scalar

**Imports needed:** `torch`, `torch.nn`, `torch.nn.functional`

In [ ]:
# YOUR IMPLEMENTATION — attempt this before looking at the solution below
# Set a timer. When it rings, stop and check the solution.

raise NotImplementedError


In [ ]:
# === SOLUTION === (only look after attempting above) ===

import torch
import torch.nn as nn
import torch.nn.functional as F


def linear_beta_schedule(timesteps: int) -> torch.Tensor:
    """Linear noise schedule from Ho et al. 2020."""
    return torch.linspace(1e-4, 0.02, timesteps)


class DDPMScheduler:
    """DDPM noise scheduler with precomputed diffusion coefficients.

    Args:
        timesteps: Total number of diffusion steps T.
    """

    def __init__(self, timesteps: int = 1000) -> None:
        self.timesteps = timesteps
        betas = linear_beta_schedule(timesteps)
        alphas = 1.0 - betas
        alpha_cumprod = torch.cumprod(alphas, dim=0)

        self.betas = betas
        self.alphas = alphas
        self.alpha_cumprod = alpha_cumprod
        self.sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod)

    def _extract(self, arr: torch.Tensor, t: torch.Tensor, x_shape: tuple) -> torch.Tensor:
        """Gather values at timesteps t and reshape for broadcasting."""
        batch_size = t.shape[0]
        out = arr.gather(0, t.cpu())  # (B,)
        # Reshape to (B, 1, 1, ...) to broadcast over spatial dims
        return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

    def forward_diffusion(
        self, x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor
    ) -> torch.Tensor:
        """Apply forward diffusion: x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1-alpha_bar_t)*noise.

        Args:
            x_0: Clean image tensor of shape (B, C, H, W).
            t: Integer timestep indices of shape (B,).
            noise: Gaussian noise tensor matching x_0 shape.

        Returns:
            Noised image x_t of same shape as x_0.
        """
        sqrt_alpha_bar = self._extract(self.sqrt_alpha_cumprod, t, x_0.shape)
        sqrt_one_minus_alpha_bar = self._extract(self.sqrt_one_minus_alpha_cumprod, t, x_0.shape)
        return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise


def training_step(
    model: nn.Module, x_0: torch.Tensor, scheduler: DDPMScheduler
) -> torch.Tensor:
    """Single DDPM training step: sample t, noise x_0, predict noise, compute MSE.

    Args:
        model: Noise prediction network accepting (x_t, t) -> predicted_noise.
        x_0: Batch of clean images (B, C, H, W).
        scheduler: DDPMScheduler instance.

    Returns:
        Scalar MSE loss between true and predicted noise.
    """
    B = x_0.shape[0]
    t = torch.randint(0, scheduler.timesteps, (B,), device=x_0.device)
    noise = torch.randn_like(x_0)
    x_t = scheduler.forward_diffusion(x_0, t, noise)
    predicted_noise = model(x_t, t)
    return F.mse_loss(predicted_noise, noise)


# --- Dummy model (3-layer conv U-Net stand-in) ---
class SimpleDummyNet(nn.Module):
    """Minimal conv net for testing: maps (B,C,H,W) + (B,) -> (B,C,H,W)."""

    def __init__(self, channels: int = 3) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, channels, 3, padding=1),
        )
        # Time embedding MLP (scalar t -> channel bias)
        self.time_mlp = nn.Linear(1, 32)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        # Simple: just run the conv net (time conditioning omitted for brevity)
        _ = self.time_mlp(t.float().unsqueeze(-1))  # (B, 32) — used in real UNet
        return self.net(x)


# --- Tests ---
scheduler = DDPMScheduler(timesteps=1000)
model = SimpleDummyNet(channels=3)

x_0 = torch.randn(4, 3, 32, 32)
t_sample = torch.randint(0, 1000, (4,))
noise = torch.randn_like(x_0)

x_t = scheduler.forward_diffusion(x_0, t_sample, noise)
assert x_t.shape == x_0.shape, f"x_t shape mismatch: {x_t.shape}"

loss = training_step(model, x_0, scheduler)
assert loss.shape == torch.Size([]), f"Loss should be scalar, got {loss.shape}"

loss.backward()

print(f"x_0 shape:  {x_0.shape}")
print(f"x_t shape:  {x_t.shape}")
print(f"Loss value: {loss.item():.4f}")
print(f"alpha_cumprod[0]:   {scheduler.alpha_cumprod[0]:.4f}  (near 1.0 = little noise)")
print(f"alpha_cumprod[999]: {scheduler.alpha_cumprod[-1]:.4f}  (near 0.0 = pure noise)")
print("Training step: OK")

## Drill 3: Data Quality Scoring Pipeline

**Time target: 35 minutes**

### Problem

Build a composable image quality scoring pipeline using only PIL, numpy, and torch.

**Class interface:**
```python
class QualityScorer:
    def __init__(self, weights: dict[str, float] | None = None) -> None: ...
    def resolution_score(self, image: Image.Image) -> float: ...   # 0.0 – 1.0
    def sharpness_score(self, image: Image.Image) -> float: ...    # 0.0 – 1.0
    def aspect_ratio_score(self, image: Image.Image) -> float: ... # 0.0 – 1.0
    def score(self, image: Image.Image) -> float: ...              # weighted composite

def filter_batch(images: list, threshold: float) -> list: ...
```

**Scoring rules:**
- `resolution_score`: 1.0 if shortest side >= 512px, else proportional (short_side / 512). Clamped to [0, 1].
- `sharpness_score`: Laplacian variance of grayscale image, normalized to [0, 1] (saturate at variance = 500).
- `aspect_ratio_score`: 1.0 if ratio in [0.5, 2.0], else linearly decays to 0 outside that range (max ratio 10.0).
- Default weights: `{"resolution": 0.3, "sharpness": 0.5, "aspect_ratio": 0.2}`

**Test with 3 synthetic images:**
1. Good image: 1024x768, random high-variance pixels
2. Bad image: 128x128, uniform gray (zero variance)
3. Extreme ratio: 100x2000

**Imports needed:** `PIL.Image`, `numpy`, `torch`

In [ ]:
# YOUR IMPLEMENTATION — attempt this before looking at the solution below
# Set a timer. When it rings, stop and check the solution.

raise NotImplementedError


In [ ]:
# === SOLUTION === (only look after attempting above) ===

import numpy as np
from PIL import Image


class QualityScorer:
    """Composable image quality scorer using sub-scores and weighted aggregation.

    Args:
        weights: Dict mapping score name to weight. Must sum to 1.0.
                 Defaults to {resolution: 0.3, sharpness: 0.5, aspect_ratio: 0.2}.
    """

    DEFAULT_WEIGHTS = {"resolution": 0.3, "sharpness": 0.5, "aspect_ratio": 0.2}

    def __init__(self, weights: dict[str, float] | None = None) -> None:
        self.weights = weights if weights is not None else self.DEFAULT_WEIGHTS
        assert abs(sum(self.weights.values()) - 1.0) < 1e-6, "Weights must sum to 1.0"

    def resolution_score(self, image: Image.Image) -> float:
        """Score based on shortest side relative to 512px minimum."""
        short_side = min(image.width, image.height)
        return min(1.0, short_side / 512.0)

    def sharpness_score(self, image: Image.Image) -> float:
        """Score based on Laplacian variance of grayscale image, saturating at 500."""
        gray = np.array(image.convert("L"), dtype=np.float32)
        # Discrete Laplacian via numpy convolution (avoids cv2 dependency)
        laplacian_kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
        from numpy.lib.stride_tricks import sliding_window_view
        # Pad then apply 3x3 kernel manually using einsum for clarity
        padded = np.pad(gray, 1, mode="reflect")
        windows = sliding_window_view(padded, (3, 3))  # (H, W, 3, 3)
        lap = np.einsum("hwij,ij->hw", windows, laplacian_kernel)
        variance = float(lap.var())
        return min(1.0, variance / 500.0)

    def aspect_ratio_score(self, image: Image.Image) -> float:
        """Score based on aspect ratio; ideal range [0.5, 2.0], penalty outside."""
        ratio = image.width / image.height
        # Normalize: ratios < 1 are inverted so both orientations treated equally
        r = max(ratio, 1.0 / ratio)  # r >= 1.0 always
        if r <= 2.0:
            return 1.0
        # Linear decay from 1.0 at r=2.0 to 0.0 at r=10.0
        return max(0.0, 1.0 - (r - 2.0) / 8.0)

    def score(self, image: Image.Image) -> float:
        """Compute weighted composite quality score in [0.0, 1.0]."""
        sub_scores = {
            "resolution": self.resolution_score(image),
            "sharpness": self.sharpness_score(image),
            "aspect_ratio": self.aspect_ratio_score(image),
        }
        return sum(self.weights[k] * v for k, v in sub_scores.items())


def filter_batch(images: list[Image.Image], threshold: float, scorer: QualityScorer | None = None) -> list[Image.Image]:
    """Return images whose composite quality score meets or exceeds threshold.

    Args:
        images: List of PIL Images.
        threshold: Minimum score in [0.0, 1.0] to pass.
        scorer: QualityScorer instance; creates default if None.

    Returns:
        Filtered list of passing images.
    """
    if scorer is None:
        scorer = QualityScorer()
    return [img for img in images if scorer.score(img) >= threshold]


# --- Test images ---
# 1. Good: large, high-frequency (sharp), normal ratio
good_arr = np.random.randint(0, 256, (768, 1024, 3), dtype=np.uint8)
good_img = Image.fromarray(good_arr)

# 2. Bad: tiny, uniform (blurry), normal ratio
bad_arr = np.full((128, 128, 3), 127, dtype=np.uint8)
bad_img = Image.fromarray(bad_arr)

# 3. Extreme aspect ratio: tall and thin
ratio_arr = np.random.randint(0, 256, (2000, 100, 3), dtype=np.uint8)
ratio_img = Image.fromarray(ratio_arr)

scorer = QualityScorer()
test_cases = [
    ("Good (1024x768, sharp)", good_img),
    ("Bad (128x128, uniform)", bad_img),
    ("Extreme ratio (100x2000)", ratio_img),
]

print(f"{'Image':<30} {'Resolution':>10} {'Sharpness':>10} {'AspectRatio':>12} {'Composite':>10}")
print("-" * 80)
for label, img in test_cases:
    r = scorer.resolution_score(img)
    s = scorer.sharpness_score(img)
    a = scorer.aspect_ratio_score(img)
    c = scorer.score(img)
    print(f"{label:<30} {r:>10.3f} {s:>10.3f} {a:>12.3f} {c:>10.3f}")

all_images = [img for _, img in test_cases]
passed = filter_batch(all_images, threshold=0.5, scorer=scorer)
print(f"\nImages passing threshold=0.5: {len(passed)}/{len(all_images)}")

## Drill 4: Configurable Noise Scheduler

**Time target: 35 minutes**

### Problem

Implement a `NoiseScheduler` class supporting three noise schedule types: linear, cosine, and sigmoid.

**Interface:**
```python
class NoiseScheduler:
    def __init__(self, num_timesteps: int, schedule_type: str) -> None: ...
    def add_noise(self, x_0: Tensor, t: Tensor, noise: Tensor) -> Tensor: ...
    def step(self, model_output: Tensor, t: int, x_t: Tensor) -> Tensor: ...
    def get_velocity(self, x_0: Tensor, noise: Tensor, t: Tensor) -> Tensor: ...
```

**Schedule definitions:**
- **Linear:** betas linearly spaced from 1e-4 to 0.02
- **Cosine:** alpha_cumprod(t) = cos((t/T + s) / (1 + s) * pi/2)^2, s=0.008; derive betas from the ratio of consecutive alphas
- **Sigmoid:** betas = sigmoid(linspace(-6, 6, T)), scaled to [1e-4, 0.02]

**Method details:**
- `add_noise`: forward diffusion reparameterization (same as Drill 2)
- `step`: DDPM reverse step — compute `x_{t-1}` from predicted noise. At t > 0, inject noise scaled by `sqrt(beta_t)`.
- `get_velocity`: v-prediction: `v = sqrt(alpha_cumprod_t) * noise - sqrt(1 - alpha_cumprod_t) * x_0`

**Test:** Plot `alpha_cumprod` vs timestep for all 3 schedule types on a single matplotlib figure. Also run a round-trip test: `add_noise` then `step` for a single t and verify shape.

**Imports needed:** `torch`, `torch.nn.functional`, `math`, `matplotlib.pyplot`

In [ ]:
# YOUR IMPLEMENTATION — attempt this before looking at the solution below
# Set a timer. When it rings, stop and check the solution.

raise NotImplementedError


In [ ]:
# === SOLUTION === (only look after attempting above) ===

import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


class NoiseScheduler:
    """Configurable DDPM noise scheduler supporting linear, cosine, and sigmoid schedules.

    Args:
        num_timesteps: Total diffusion steps T.
        schedule_type: One of 'linear', 'cosine', 'sigmoid'.
    """

    def __init__(self, num_timesteps: int, schedule_type: str) -> None:
        self.T = num_timesteps
        self.schedule_type = schedule_type
        self.betas = self._make_betas()
        self._precompute()

    def _make_betas(self) -> torch.Tensor:
        if self.schedule_type == "linear":
            return torch.linspace(1e-4, 0.02, self.T)

        elif self.schedule_type == "cosine":
            # Compute alpha_cumprod directly via cosine formula, then derive betas
            s = 0.008
            steps = torch.arange(self.T + 1, dtype=torch.float32)
            f = torch.cos((steps / self.T + s) / (1 + s) * math.pi / 2) ** 2
            alpha_cumprod = f / f[0]  # normalize so alpha_cumprod[0] = 1
            # betas from ratio of consecutive alphas
            betas = 1.0 - alpha_cumprod[1:] / alpha_cumprod[:-1]
            return betas.clamp(0.0, 0.999)

        elif self.schedule_type == "sigmoid":
            raw = torch.sigmoid(torch.linspace(-6, 6, self.T))
            # Scale raw [sigmoid(-6), sigmoid(6)] -> [beta_min, beta_max]
            beta_min, beta_max = 1e-4, 0.02
            raw_min, raw_max = raw.min(), raw.max()
            return (raw - raw_min) / (raw_max - raw_min) * (beta_max - beta_min) + beta_min

        else:
            raise ValueError(f"Unknown schedule_type: {self.schedule_type!r}")

    def _precompute(self) -> None:
        alphas = 1.0 - self.betas
        self.alpha_cumprod = torch.cumprod(alphas, dim=0)
        self.sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod)
        self.sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - self.alpha_cumprod)

    def _extract(self, arr: torch.Tensor, t: torch.Tensor | int, shape: tuple) -> torch.Tensor:
        """Gather values at t and broadcast to match spatial dims."""
        if isinstance(t, int):
            t = torch.tensor([t])
        out = arr.gather(0, t.long().cpu())
        return out.reshape(t.shape[0], *((1,) * (len(shape) - 1)))

    def add_noise(
        self, x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor
    ) -> torch.Tensor:
        """Forward diffusion: x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1-alpha_bar_t)*noise."""
        s = self._extract(self.sqrt_alpha_cumprod, t, x_0.shape)
        sm = self._extract(self.sqrt_one_minus_alpha_cumprod, t, x_0.shape)
        return s * x_0 + sm * noise

    def step(
        self, model_output: torch.Tensor, t: int, x_t: torch.Tensor
    ) -> torch.Tensor:
        """DDPM reverse step: predict x_{t-1} from predicted noise and x_t.

        Args:
            model_output: Predicted noise epsilon_theta(x_t, t), same shape as x_t.
            t: Current scalar timestep (int).
            x_t: Noisy sample at timestep t.

        Returns:
            Denoised sample x_{t-1}.
        """
        beta_t = self.betas[t]
        sqrt_alpha_t = torch.sqrt(1.0 - beta_t)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_cumprod[t]

        # Predicted x_0 coefficient = 1/sqrt(alpha_t)
        # Mean of posterior: mu = (1/sqrt(alpha_t)) * (x_t - beta_t/sqrt(1-alpha_bar_t) * eps)
        coef = beta_t / sqrt_one_minus_alpha_bar_t
        mean = (1.0 / sqrt_alpha_t) * (x_t - coef * model_output)

        if t > 0:
            noise = torch.randn_like(x_t)
            return mean + torch.sqrt(beta_t) * noise
        return mean

    def get_velocity(
        self, x_0: torch.Tensor, noise: torch.Tensor, t: torch.Tensor
    ) -> torch.Tensor:
        """V-prediction target: v = sqrt(alpha_bar_t)*noise - sqrt(1-alpha_bar_t)*x_0."""
        s = self._extract(self.sqrt_alpha_cumprod, t, x_0.shape)
        sm = self._extract(self.sqrt_one_minus_alpha_cumprod, t, x_0.shape)
        return s * noise - sm * x_0


# --- Visualization: compare alpha_cumprod across schedules ---
T = 1000
schedules = ["linear", "cosine", "sigmoid"]
schedulers = {s: NoiseScheduler(T, s) for s in schedules}

fig, ax = plt.subplots(figsize=(8, 4))
for name, sched in schedulers.items():
    ax.plot(sched.alpha_cumprod.numpy(), label=name)
ax.set_xlabel("Timestep t")
ax.set_ylabel(r"$\bar{\alpha}_t$ (alpha_cumprod)")
ax.set_title("Noise Schedule Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Round-trip test ---
sched = schedulers["cosine"]
x_0 = torch.randn(2, 3, 32, 32)
noise = torch.randn_like(x_0)
t = torch.tensor([500, 500])

x_t = sched.add_noise(x_0, t, noise)
assert x_t.shape == x_0.shape, f"Shape mismatch: {x_t.shape}"

# step with ground-truth noise (ideal model output)
x_prev = sched.step(model_output=noise, t=500, x_t=x_t)
assert x_prev.shape == x_0.shape

v = sched.get_velocity(x_0, noise, t)
assert v.shape == x_0.shape

print(f"x_t shape:   {x_t.shape}")
print(f"x_prev shape: {x_prev.shape}")
print(f"v shape:      {v.shape}")
print("All schedule types and methods: OK")

## Drill 5: Embedding-Based Batch Deduplication

**Time target: 35 minutes**

### Problem

Implement an `EmbeddingDeduplicator` for finding and removing near-duplicate images using cosine similarity.

**Interface:**
```python
class EmbeddingDeduplicator:
    def __init__(self, similarity_threshold: float = 0.95) -> None: ...
    def embed_batch(self, images: list[Tensor], model: nn.Module) -> Tensor: ...
    def build_similarity_matrix(self, embeddings: Tensor) -> Tensor: ...
    def find_duplicate_groups(self, sim_matrix: Tensor) -> list[set[int]]: ...
    def deduplicate(self, embeddings: Tensor) -> list[int]: ...
```

**Method details:**
- `embed_batch`: Run model on each image, L2-normalize the output embeddings, return `(N, D)` tensor
- `build_similarity_matrix`: Compute cosine similarity via matmul of L2-normalized embeddings → `(N, N)` tensor
- `find_duplicate_groups`: Connected components — two indices are in the same group if their similarity >= threshold. Use union-find.
- `deduplicate`: Returns sorted list of indices to keep (first index encountered in each group)

**Test setup:** Create 10 synthetic embeddings where:
- Indices 0, 3, 7 are near-duplicates of each other
- Indices 2, 5 are near-duplicates of each other
- All other indices are unique

Expected kept indices: `{0, 1, 2, 4, 6, 8, 9}` (one per group, first occurrence)

**Imports needed:** `torch`, `torch.nn`

In [ ]:
# YOUR IMPLEMENTATION — attempt this before looking at the solution below
# Set a timer. When it rings, stop and check the solution.

raise NotImplementedError


In [ ]:
# === SOLUTION === (only look after attempting above) ===

import torch
import torch.nn as nn


class UnionFind:
    """Path-compressed union-find for connected components."""

    def __init__(self, n: int) -> None:
        self.parent = list(range(n))

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]  # path compression
            x = self.parent[x]
        return x

    def union(self, x: int, y: int) -> None:
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[max(rx, ry)] = min(rx, ry)  # smaller root wins


class EmbeddingDeduplicator:
    """Finds and removes near-duplicate images using cosine similarity on embeddings.

    Args:
        similarity_threshold: Cosine similarity >= this value → considered duplicates.
    """

    def __init__(self, similarity_threshold: float = 0.95) -> None:
        self.threshold = similarity_threshold

    def embed_batch(
        self, images: list[torch.Tensor], model: nn.Module
    ) -> torch.Tensor:
        """Embed images and return L2-normalized embeddings of shape (N, D).

        Args:
            images: List of N image tensors (each can be any shape accepted by model).
            model: Embedding model returning (1, D) or (D,) per image.

        Returns:
            L2-normalized embeddings tensor of shape (N, D).
        """
        model.eval()
        embeddings = []
        with torch.no_grad():
            for img in images:
                emb = model(img.unsqueeze(0))  # (1, D)
                embeddings.append(emb.squeeze(0))
        stacked = torch.stack(embeddings, dim=0)  # (N, D)
        return nn.functional.normalize(stacked, p=2, dim=1)

    def build_similarity_matrix(self, embeddings: torch.Tensor) -> torch.Tensor:
        """Compute all-pairs cosine similarity.

        Args:
            embeddings: L2-normalized tensor of shape (N, D).

        Returns:
            Symmetric similarity matrix of shape (N, N), values in [-1, 1].
        """
        # For L2-normalized vectors: cosine_sim = dot product
        return embeddings @ embeddings.T

    def find_duplicate_groups(self, sim_matrix: torch.Tensor) -> list[set[int]]:
        """Find connected components of near-duplicates using union-find.

        Args:
            sim_matrix: (N, N) cosine similarity matrix.

        Returns:
            List of sets, each set containing indices in the same duplicate group.
        """
        N = sim_matrix.shape[0]
        uf = UnionFind(N)

        # Find pairs above threshold (upper triangle only to avoid double-processing)
        above = (sim_matrix >= self.threshold).triu(diagonal=1)
        pairs = above.nonzero(as_tuple=False)  # (K, 2)

        for i, j in pairs.tolist():
            uf.union(i, j)

        # Collect groups by root
        from collections import defaultdict
        groups: dict[int, set[int]] = defaultdict(set)
        for idx in range(N):
            groups[uf.find(idx)].add(idx)

        return list(groups.values())

    def deduplicate(self, embeddings: torch.Tensor) -> list[int]:
        """Return sorted list of indices to keep (one per duplicate group).

        Args:
            embeddings: (N, D) embedding tensor (need not be L2-normalized beforehand).

        Returns:
            Sorted list of kept indices.
        """
        normed = nn.functional.normalize(embeddings, p=2, dim=1)
        sim_matrix = self.build_similarity_matrix(normed)
        groups = self.find_duplicate_groups(sim_matrix)
        # Keep the lowest index in each group
        return sorted(min(group) for group in groups)


# --- Test: build synthetic embeddings with controlled near-duplicates ---
torch.manual_seed(42)
N, D = 10, 128

# Start with random unique embeddings
embeddings = torch.randn(N, D)

# Make indices 0, 3, 7 near-duplicates: copy index 0, add tiny noise
embeddings[3] = embeddings[0] + torch.randn(D) * 0.001
embeddings[7] = embeddings[0] + torch.randn(D) * 0.001

# Make indices 2, 5 near-duplicates
embeddings[5] = embeddings[2] + torch.randn(D) * 0.001

# L2-normalize all
embeddings = nn.functional.normalize(embeddings, p=2, dim=1)

deduplicator = EmbeddingDeduplicator(similarity_threshold=0.95)

sim_matrix = deduplicator.build_similarity_matrix(embeddings)
groups = deduplicator.find_duplicate_groups(sim_matrix)
kept = deduplicator.deduplicate(embeddings)

print("Duplicate groups found:")
for g in sorted(groups, key=min):
    print(f"  {sorted(g)}")

print(f"\nIndices to keep: {kept}")
print(f"Expected:        {sorted({0, 1, 2, 4, 6, 8, 9})}")

assert set(kept) == {0, 1, 2, 4, 6, 8, 9}, f"Unexpected kept indices: {kept}"
print("\nDeduplication: OK")

## Checkpoint: Implementation Tips

### Time management

- **Read the full problem before writing a single line of code.** Identify the data types, shapes, and edge cases first.
- Write class/function signatures and docstrings first — this forces you to think about the interface before the implementation.
- Test incrementally as you go. Don't wait until the end to run anything; catch shape errors early.
- If you're stuck on one part, write a stub that returns a plausible fake value and move on. Return to it.
- Reserve the last 5 minutes to clean up naming, add a quick assert, and explain your code aloud.

### Common mistakes

- **Forgetting `math.sqrt`** in scaled dot-product attention (or using `** 0.5` inconsistently).
- **Off-by-one in timestep indexing** — be explicit about whether t=0 is the clean image or the first noising step.
- **Not handling batch dimensions** — always confirm shapes at each step; use `assert tensor.shape == expected`.
- **Using numpy where PyTorch is expected** — if the problem says PyTorch, stay in torch throughout (gradients won't flow through numpy).
- **Forgetting `.contiguous()`** after `.transpose()` before `.view()` — this causes silent errors in reshape.
- **Not L2-normalizing before cosine similarity** — just doing dot products on unnormalized vectors gives wrong results.

### What matters in practice

- **Clean code structure:** Classes with clear `__init__` / `forward` / `step` separation. Consistent naming. Type hints on every function signature.
- **Handling edge cases:** Zero-size batches, t=0 in the reverse process, mask shapes that broadcast correctly.
- **PyTorch fluency:** Using `gather`, `masked_fill`, `einsum`, `triu`, `nonzero` naturally — not reinventing them with loops.
- **Explaining tradeoffs verbally:** Why cosine vs L2 similarity? Why union-find over naive pairwise? Why v-prediction vs noise prediction? Have answers ready.
- **Asking clarifying questions before diving in:** "Should the mask be boolean or float? Is this batched or single-image? Do you want gradient flow through the embeddings?"